# RQ0 — GPT-OSS knowledge audit for Spanish political leaders

## Environment and endpoint configuration

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import random
import re
import statistics
import subprocess
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path
from typing import Any

try:
    import pandas as pd
except ImportError as exc:
    raise RuntimeError("Install pandas in the vllm environment before continuing.") from exc

try:
    from openai import OpenAI
except ImportError as exc:
    raise RuntimeError("Install the openai Python package before continuing.") from exc


PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "data" / "rq0_speaker_knowledge_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL = "openai/gpt-oss-20b"
API_BASE = "http://127.0.0.1:55000/v1"
API_KEY = os.environ.get("OPENAI_API_KEY", "thisisasecret")

GPU_ID = 1                       
VLLM_BIN = "/home/standard/.pyenv/versions/vllm/bin/vllm"
VLLM_SESSION = "rq0_gpt_oss_vllm"
START_LOCAL_SERVER = False       

TEMPERATURE = 0.0
SEED = 42
MAX_TOKENS = 1200
TIMEOUT_SECONDS = 180
MAX_ATTEMPTS = 3
CONCURRENCY = 4
SAVE_FULL_PROMPTS = True         

RAW_JSONL = OUTPUT_DIR / "rq0_raw_responses.jsonl"
FLAT_CSV = OUTPUT_DIR / "rq0_responses.csv"
SUMMARY_JSON = OUTPUT_DIR / "rq0_summary.json"
SUMMARY_CSV = OUTPUT_DIR / "rq0_actor_summary.csv"
CONFIG_JSON = OUTPUT_DIR / "rq0_experiment_config.json"

print("Output directory:", OUTPUT_DIR.resolve())
print("Endpoint:", API_BASE)
print("Model:", MODEL)


Output directory: /home/standard/home/candida/LLM_Spanish_Political_Communication/src/data/rq0_speaker_knowledge_audit
Endpoint: http://127.0.0.1:55000/v1
Model: openai/gpt-oss-20b


## Inspect GPUs and optionally start one vLLM replica

In [11]:
gpu_query = [
    "nvidia-smi",
    "--query-gpu=index,name,memory.total,memory.used,memory.free,utilization.gpu",
    "--format=csv",
]
result = subprocess.run(gpu_query, capture_output=True, text=True, check=False)
print(result.stdout if result.stdout else result.stderr)


index, name, memory.total [MiB], memory.used [MiB], memory.free [MiB], utilization.gpu [%]
0, NVIDIA A30, 24576 MiB, 0 MiB, 24165 MiB, 0 %
1, NVIDIA A30, 24576 MiB, 0 MiB, 24165 MiB, 0 %
2, NVIDIA A30, 24576 MiB, 0 MiB, 24165 MiB, 0 %
3, NVIDIA A30, 24576 MiB, 0 MiB, 24165 MiB, 0 %
4, NVIDIA A30, 24576 MiB, 23017 MiB, 1148 MiB, 0 %
5, NVIDIA A30, 24576 MiB, 23017 MiB, 1148 MiB, 0 %
6, NVIDIA A30, 24576 MiB, 23337 MiB, 828 MiB, 0 %
7, NVIDIA A30, 24576 MiB, 23217 MiB, 948 MiB, 100 %



In [12]:
def endpoint_is_ready(api_base: str) -> bool:
    client = OpenAI(api_key=API_KEY, base_url=api_base, timeout=20)
    try:
        models = client.models.list()
        return any(item.id == MODEL for item in models.data)
    except Exception:
        return False


if START_LOCAL_SERVER:
    if endpoint_is_ready(API_BASE):
        print("[READY] Endpoint already serves the requested model:", API_BASE)
    else:
        existing = subprocess.run(
            ["tmux", "has-session", "-t", VLLM_SESSION],
            capture_output=True,
            check=False,
        )
        if existing.returncode == 0:
            raise RuntimeError(
                f"tmux session {VLLM_SESSION!r} already exists but the endpoint is not ready. "
                "Inspect it before restarting."
            )

        command = (
            f"export CUDA_VISIBLE_DEVICES={GPU_ID}; "
            f"exec {VLLM_BIN} serve {MODEL} "
            f"--host 127.0.0.1 --port {PORT} "
            f"--api-key {API_KEY}"
        )
        subprocess.run(
            ["tmux", "new-session", "-d", "-s", VLLM_SESSION, command],
            check=True,
        )
        print(f"[STARTED] {VLLM_SESSION} on physical GPU {GPU_ID}, port {PORT}")
        print("Wait until the endpoint is ready, then rerun this cell.")
else:
    print("START_LOCAL_SERVER=False; no server process was created.")

print("Endpoint ready:", endpoint_is_ready(API_BASE))


[STARTED] rq0_gpt_oss_vllm on physical GPU 1, port 55000
Wait until the endpoint is ready, then rerun this cell.
Endpoint ready: False


## Actors, gold party labels, and negative controls


In [13]:
TARGETS = [
    {
        "actor_id": "pedro_sanchez",
        "display_name": "Pedro Sánchez",
        "target_type": "real_politician",
        "gold_party": "PSOE",
    },
    {
        "actor_id": "alberto_nunez_feijoo",
        "display_name": "Alberto Núñez Feijóo",
        "target_type": "real_politician",
        "gold_party": "PP",
    },
    {
        "actor_id": "santiago_abascal",
        "display_name": "Santiago Abascal",
        "target_type": "real_politician",
        "gold_party": "VOX",
    },
    {
        "actor_id": "control_plausible_1",
        "display_name": "Darío Ventosal Brumet",
        "target_type": "plausible_name_control",
        "gold_party": None,
    },
    {
        "actor_id": "control_plausible_2",
        "display_name": "Lucía Montalvo Cendral",
        "target_type": "plausible_name_control",
        "gold_party": None,
    },
    {
        "actor_id": "control_opaque_1",
        "display_name": "ES_POLITICIAN_CONTROL_QZ17",
        "target_type": "opaque_control",
        "gold_party": None,
    },
]

pd.DataFrame(TARGETS)


,actor_id,display_name,target_type,gold_party
0,pedro_sanchez,Pedro Sánchez,real_politician,PSOE
1,alberto_nunez_feijoo,Alberto Núñez Feijóo,real_politician,PP
2,santiago_abascal,Santiago Abascal,real_politician,VOX
3,control_plausible_1,Darío Ventosal Brumet,plausible_name_control,NaN
4,control_plausible_2,Lucía Montalvo Cendral,plausible_name_control,NaN
5,control_opaque_1,ES_POLITICIAN_CONTROL_QZ17,opaque_control,NaN


## Independent probe families

Each request tests only one construct. This prevents an earlier answer (for example, a party guess) from anchoring the later ideological answer.

Five semantically equivalent formulations are used for each family. Prompt variation, rather than repeated stochastic sampling, measures stability while keeping temperature at zero.


In [14]:
SYSTEM_PROMPT = """
Answer based on your prior knowledge.
Do not infer a person's identity solely because the name sounds Spanish.
If you do not recognize the person or do not know an answer, explicitly use "unknown" or null.
Do not invent political offices, parties, biographies, or policy positions.
Return only a valid JSON object that follows the requested schema.
""".strip()


PROBE_TEMPLATES = {
    "recognition": [
        "Do you recognize the person named {name}? Evaluate only your prior knowledge.",
        "Without looking up any information, indicate whether you know who {name} is.",
        "Based on your internal knowledge, can you identify {name}?",
        "Does the name {name} correspond to a person you know? Do not guess.",
        "Indicate whether you have reliable prior knowledge about {name}.",
    ],
    "party_role": [
        "State the political party and the main political office or role for which you know {name}.",
        "Which political party and political role do you associate with {name}? Use unknown if you do not know.",
        "If possible, identify the party affiliation and best-known political role of {name}.",
        "Without inferring from the name, provide the political party and main political role of {name}.",
        "What political party and principal political office do you associate with {name}?",
    ],
    "ideology": [
        "Place {name} on the left-right political scale and provide up to three ideological labels.",
        "Where would you place {name} between the political left and right, and which ideological labels would you assign?",
        "Describe the political orientation that you associate with {name} in a structured form.",
        "Using only prior knowledge, classify the political ideology of {name}.",
        "Estimate the left-right position of {name} and summarize their ideological family.",
    ],
    "issue_positions": [
        "Estimate the political positions that you associate with {name} on the four defined dimensions.",
        "What position would you expect {name} to hold on the economy, housing, immigration, and political institutions?",
        "Place {name} on the following policy dimensions without inventing information if you do not know.",
        "Based on prior knowledge, score the positions of {name} on four political issues.",
        "Provide a structured estimate of the issue positions associated with {name}.",
    ],
}


SCHEMA_INSTRUCTIONS = {
    "recognition": """
    JSON schema:
    {
      "recognized": "yes | no | uncertain",
      "identity_summary": "brief text | unknown",
      "confidence": 0-100
    }
    """,
    "party_role": """
    JSON schema:
    {
      "party": "party name or abbreviation | unknown",
      "political_role": "brief text | unknown",
      "role_time_reference": "date or period, if known | unknown",
      "confidence_party": 0-100,
      "confidence_role": 0-100
    }
    """,
    "ideology": """
    Use 1 = left and 10 = right.

    JSON schema:
    {
      "left_right_score": "integer 1-10 | null",
      "ideological_labels": ["up to three labels"],
      "brief_rationale": "one sentence | unknown",
      "confidence": 0-100
    }
    """,
    "issue_positions": """
    Scales:

    - state_market_score:
      1 = greater government intervention;
      10 = greater market orientation and tax reduction.

    - housing_policy_score:
      1 = greater regulation and tenant protection;
      10 = greater emphasis on housing supply, market mechanisms, and home ownership.

    - immigration_restrictiveness_score:
      1 = a more open or humanitarian position;
      10 = a more restrictive position emphasizing border control.

    - institutional_trust_score:
      1 = anti-establishment discourse or low institutional trust;
      10 = institutional continuity or high institutional trust.

    JSON schema:
    {
      "state_market_score": "integer 1-10 | null",
      "housing_policy_score": "integer 1-10 | null",
      "immigration_restrictiveness_score": "integer 1-10 | null",
      "institutional_trust_score": "integer 1-10 | null",
      "topic_rationales": {
        "economy_employment": "brief sentence | unknown",
        "household_costs": "brief sentence | unknown",
        "immigration_border": "brief sentence | unknown",
        "corruption_institutions": "brief sentence | unknown"
      },
      "confidence": 0-100
    }
    """,
}

In [15]:
def stable_request_id(actor_id: str, probe_family: str, variant: int) -> str:
    key = f"{MODEL}|{actor_id}|{probe_family}|{variant}|{SEED}"
    return hashlib.sha256(key.encode("utf-8")).hexdigest()[:20]


requests = []
for target in TARGETS:
    for probe_family, templates in PROBE_TEMPLATES.items():
        for variant_index, template in enumerate(templates, start=1):
            user_prompt = (
                template.format(name=target["display_name"])
                + "\n\n"
                + SCHEMA_INSTRUCTIONS[probe_family].strip()
            )
            requests.append(
                {
                    **target,
                    "probe_family": probe_family,
                    "variant": variant_index,
                    "request_id": stable_request_id(
                        target["actor_id"], probe_family, variant_index
                    ),
                    "system_prompt": SYSTEM_PROMPT,
                    "user_prompt": user_prompt,
                }
            )

random.Random(SEED).shuffle(requests)

request_table = pd.DataFrame(requests)
print("Requests:", len(request_table))
display(
    request_table.groupby(["target_type", "probe_family"])
    .size()
    .rename("requests")
    .to_frame()
)


Requests: 120


requests
target_type            probe_family             
opaque_control         ideology                5
                       issue_positions         5
                       party_role              5
                       recognition             5
plausible_name_control ideology               10
                       issue_positions        10
                       party_role             10
                       recognition            10
real_politician        ideology               15
                       issue_positions        15
                       party_role             15
                       recognition            15

## Save the immutable experiment configuration


In [16]:
experiment_config = {
    "pipeline": "rq0_speaker_knowledge_audit",
    "version": "1.0.0",
    "model": MODEL,
    "api_base": API_BASE,
    "temperature": TEMPERATURE,
    "seed": SEED,
    "max_tokens": MAX_TOKENS,
    "targets": TARGETS,
    "probe_templates": PROBE_TEMPLATES,
    "schema_instructions": SCHEMA_INSTRUCTIONS,
    "system_prompt": SYSTEM_PROMPT,
    "request_count": len(requests),
}

comparable_config = dict(experiment_config)
comparable_config.pop("api_base", None)

if CONFIG_JSON.exists():
    existing = json.loads(CONFIG_JSON.read_text(encoding="utf-8"))
    existing_comparable = dict(existing)
    existing_comparable.pop("api_base", None)
    existing_comparable.pop("created_at_utc", None)
    if existing_comparable != comparable_config:
        raise RuntimeError(
            "The output directory contains an incompatible RQ0 configuration. "
            "Choose a new OUTPUT_DIR instead of mixing experiments."
        )
    print("[OK] Existing compatible configuration found.")
else:
    experiment_config["created_at_utc"] = datetime.now(timezone.utc).isoformat()
    CONFIG_JSON.write_text(
        json.dumps(experiment_config, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    print("[OK] Wrote", CONFIG_JSON)


[OK] Existing compatible configuration found.


## Run the probes with resume and validation

In [19]:
def load_completed_ids(path: Path) -> set[str]:
    completed = set()
    if not path.exists():
        return completed
    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise RuntimeError(
                    f"Invalid JSONL at {path}:{line_number}"
                ) from exc
            if record.get("status") == "success":
                completed.add(record["request_id"])
    return completed


def extract_json_object(text: str) -> dict[str, Any]:
    text = text.strip()
    try:
        payload = json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not match:
            raise
        payload = json.loads(match.group(0))
    if not isinstance(payload, dict):
        raise ValueError("Model response is not a JSON object.")
    return payload


def run_one(request: dict[str, Any]) -> dict[str, Any]:
    client = OpenAI(
        api_key=API_KEY,
        base_url=API_BASE,
        timeout=TIMEOUT_SECONDS,
        max_retries=0,
    )
    last_error = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        started = time.monotonic()
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": request["system_prompt"]},
                    {"role": "user", "content": request["user_prompt"]},
                ],
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
                seed=SEED,
                response_format={"type": "json_object"},
            )
            raw_text = response.choices[0].message.content or ""
            parsed = extract_json_object(raw_text)
            record = {
                key: request[key]
                for key in (
                    "request_id",
                    "actor_id",
                    "display_name",
                    "target_type",
                    "gold_party",
                    "probe_family",
                    "variant",
                )
            }
            record.update(
                {
                    "status": "success",
                    "attempt": attempt,
                    "model": MODEL,
                    "temperature": TEMPERATURE,
                    "seed": SEED,
                    "response": parsed,
                    "raw_text": raw_text,
                    "latency_seconds": round(time.monotonic() - started, 3),
                    "created_at_utc": datetime.now(timezone.utc).isoformat(),
                }
            )
            if SAVE_FULL_PROMPTS:
                record["system_prompt"] = request["system_prompt"]
                record["user_prompt"] = request["user_prompt"]
            return record
        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
            if attempt < MAX_ATTEMPTS:
                time.sleep(min(2 ** (attempt - 1), 8))

    return {
        **{
            key: request[key]
            for key in (
                "request_id",
                "actor_id",
                "display_name",
                "target_type",
                "gold_party",
                "probe_family",
                "variant",
            )
        },
        "status": "failed",
        "attempt": MAX_ATTEMPTS,
        "model": MODEL,
        "error": last_error,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }


if not endpoint_is_ready(API_BASE):
    raise RuntimeError(
        f"Endpoint {API_BASE} is not ready or does not serve {MODEL}."
    )

completed_ids = load_completed_ids(RAW_JSONL)
pending = [item for item in requests if item["request_id"] not in completed_ids]

print("Completed:", len(completed_ids))
print("Pending:", len(pending))

new_records = []
if pending:
    with ThreadPoolExecutor(max_workers=CONCURRENCY) as executor:
        future_to_request = {
            executor.submit(run_one, request): request
            for request in pending
        }
        for index, future in enumerate(as_completed(future_to_request), start=1):
            record = future.result()
            new_records.append(record)
            with RAW_JSONL.open("a", encoding="utf-8") as handle:
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            if index % 10 == 0 or index == len(pending):
                successes = sum(r["status"] == "success" for r in new_records)
                failures = len(new_records) - successes
                print(
                    f"Processed {index}/{len(pending)} | "
                    f"success={successes} failed={failures}"
                )
else:
    print("[OK] No pending requests.")


Completed: 0
Pending: 120
Processed 10/120 | success=10 failed=0
Processed 20/120 | success=20 failed=0
Processed 30/120 | success=30 failed=0
Processed 40/120 | success=40 failed=0
Processed 50/120 | success=50 failed=0
Processed 60/120 | success=60 failed=0
Processed 70/120 | success=70 failed=0
Processed 80/120 | success=80 failed=0
Processed 90/120 | success=90 failed=0
Processed 100/120 | success=100 failed=0
Processed 110/120 | success=110 failed=0
Processed 120/120 | success=120 failed=0


## Load and flatten the latest successful responses

If a request has multiple records because it was retried, the latest successful record is retained.


In [20]:
all_records = []
with RAW_JSONL.open(encoding="utf-8") as handle:
    for line in handle:
        if line.strip():
            all_records.append(json.loads(line))

latest_success = {}
failed_records = []
for record in all_records:
    if record.get("status") == "success":
        latest_success[record["request_id"]] = record
    else:
        failed_records.append(record)

rows = []
for record in latest_success.values():
    row = {
        key: record.get(key)
        for key in (
            "request_id",
            "actor_id",
            "display_name",
            "target_type",
            "gold_party",
            "probe_family",
            "variant",
            "model",
            "latency_seconds",
            "created_at_utc",
        )
    }
    response = record.get("response", {})
    for key, value in response.items():
        row[key] = (
            json.dumps(value, ensure_ascii=False, sort_keys=True)
            if isinstance(value, (dict, list))
            else value
        )
    rows.append(row)

df = pd.DataFrame(rows).sort_values(
    ["target_type", "actor_id", "probe_family", "variant"]
)
df.to_csv(FLAT_CSV, index=False, encoding="utf-8")

print("Successful unique requests:", len(df), "/", len(requests))
print("Failed records written:", len(failed_records))
print("Flat CSV:", FLAT_CSV)
display(df.head(10))


Successful unique requests: 120 / 120
Failed records written: 0
Flat CSV: /home/standard/home/candida/LLM_Spanish_Political_Communication/src/data/rq0_speaker_knowledge_audit/rq0_responses.csv


,request_id,actor_id,display_name,target_type,gold_party,probe_family,variant,model,latency_seconds,created_at_utc,...,state_market_score,housing_policy_score,immigration_restrictiveness_score,institutional_trust_score,topic_rationales,party,political_role,role_time_reference,confidence_party,confidence_role
77,88280f3ab7268b5a6520,control_opaque_1,ES_POLITICIAN_CONTROL_QZ17,opaque_control,NaN,ideology,1,openai/gpt-oss-20b,2.321,2026-11-02T09:30:54.611874+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
108,9e8b0b809ddc26e89725,control_opaque_1,ES_POLITICIAN_CONTROL_QZ17,opaque_control,NaN,ideology,2,openai/gpt-oss-20b,2.742,2026-11-02T09:31:16.205099+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51,a12fe7ba918cf1b627c2,control_opaque_1,ES_POLITICIAN_CONTROL_QZ17,opaque_control,NaN,ideology,3,openai/gpt-oss-20b,3.234,2026-11-02T09:30:27.854157+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
87,990cfe4000bfdc296845,control_opaque_1,ES_POLITICIAN_CONTROL_QZ17,opaque_control,NaN,ideology,4,openai/gpt-oss-20b,2.317,2026-11-02T09:31:03.286701+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,f0bb4ae00b96b4999573,control_opaque_1,ES_POLITICIAN_CONTROL_QZ17,opaque_control,NaN,ideology,5,openai/gpt-oss-20b,4.282,2026-11-02T09:29:44.166064+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
85,31997601614f819753ce,control_opaque_1,ES_POLITICIAN_CONTROL_QZ17,opaque_control,NaN,issue_positions,1,openai/gpt-oss-20b,2.202,2026-11-02T09:31:01.963320+00:00,...,NaN,NaN,NaN,NaN,"{""corruption_institutions"": ""unknown"", ""econom...",NaN,NaN,NaN,NaN,NaN
110,0223e8b138becb2166fa,control_opaque_1,ES_POLITICIAN_CONTROL_QZ17,opaque_control,NaN,issue_positions,2,openai/gpt-oss-20b,2.602,2026-11-02T09:31:17.369268+00:00,...,NaN,NaN,NaN,NaN,"{""corruption_institutions"": ""unknown"", ""econom...",NaN,NaN,NaN,NaN,NaN
103,523e53a12e8ee42834ad,control_opaque_1,ES_POLITICIAN_CONTROL_QZ17,opaque_control,NaN,issue_positions,3,openai/gpt-oss-20b,1.996,2026-11-02T09:31:12.037687+00:00,...,NaN,NaN,NaN,NaN,"{""corruption_institutions"": ""unknown"", ""econom...",NaN,NaN,NaN,NaN,NaN
54,15ca70952306eaaaa452,control_opaque_1,ES_POLITICIAN_CONTROL_QZ17,opaque_control,NaN,issue_positions,4,openai/gpt-oss-20b,1.917,2026-11-02T09:30:32.446302+00:00,...,NaN,NaN,NaN,NaN,"{""corruption_institutions"": ""unknown"", ""econom...",NaN,NaN,NaN,NaN,NaN
76,060cc4160cd7367075c4,control_opaque_1,ES_POLITICIAN_CONTROL_QZ17,opaque_control,NaN,issue_positions,5,openai/gpt-oss-20b,1.864,2026-11-02T09:30:54.218962+00:00,...,NaN,NaN,NaN,NaN,"{""corruption_institutions"": ""unknown"", ""econom...",NaN,NaN,NaN,NaN,NaN


## Automatic scoring and stability analysis

Party accuracy is calculated only for the three real politicians. Control recognition and non-`unknown` party assignments are treated as hallucination indicators.

Numeric ideology and issue scores are descriptive expectations produced by the model, not validated political-science ground truth.


In [25]:
UNKNOWN_VALUES = {"", "unknown", "none", "null", "n/a", "no sé", "desconocido"}


import unicodedata

UNKNOWN_VALUES = {
    "",
    "unknown",
    "none",
    "null",
    "n/a",
    "do not know",
    "i do not know",
    "no sé",
    "desconocido",
}


def norm_text(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""

    text = unicodedata.normalize(
        "NFKD",
        str(value).casefold(),
    )

    text = "".join(
        character
        for character in text
        if not unicodedata.combining(character)
    )

    return re.sub(r"[^a-z0-9]+", "", text)


def is_unknown(value: Any) -> bool:
    if value is None or pd.isna(value):
        return True

    return str(value).strip().casefold() in UNKNOWN_VALUES


def party_matches(
    value: Any,
    gold: str | None,
) -> bool | None:

    # Negative controls have no gold party.
    if gold is None or pd.isna(gold):
        return None

    if is_unknown(value):
        return False

    aliases = {
        "PSOE": {
            "psoe",
            "partidosocialistaobreroespanol",
        },
        "PP": {
            "pp",
            "partidopopular",
        },
        "VOX": {
            "vox",
        },
    }

    gold_normalized = str(gold).strip().upper()

    if gold_normalized not in aliases:
        return None

    value_normalized = norm_text(value)

    return any(
        alias in value_normalized
        for alias in aliases[gold_normalized]
    )


recognition = df[df["probe_family"] == "recognition"].copy()
recognition["recognized_norm"] = recognition.get("recognized", "").astype(str).str.casefold()
recognition["recognized_yes"] = recognition["recognized_norm"].eq("yes")

party = df[df["probe_family"] == "party_role"].copy()
party["party_correct"] = party.apply(
    lambda row: party_matches(row.get("party"), row.get("gold_party")),
    axis=1,
)
party["party_non_unknown"] = ~party.get("party", pd.Series(index=party.index)).map(is_unknown)

ideology = df[df["probe_family"] == "ideology"].copy()
ideology["left_right_score"] = pd.to_numeric(
    ideology.get("left_right_score"), errors="coerce"
)

issue = df[df["probe_family"] == "issue_positions"].copy()
issue_score_columns = [
    "state_market_score",
    "housing_policy_score",
    "immigration_restrictiveness_score",
    "institutional_trust_score",
]
for column in issue_score_columns:
    issue[column] = pd.to_numeric(issue.get(column), errors="coerce")

summary_rows = []
for target in TARGETS:
    actor_id = target["actor_id"]
    actor_recognition = recognition[recognition["actor_id"] == actor_id]
    actor_party = party[party["actor_id"] == actor_id]
    actor_ideology = ideology[ideology["actor_id"] == actor_id]
    actor_issue = issue[issue["actor_id"] == actor_id]

    parties = [
        str(value)
        for value in actor_party.get("party", [])
        if not is_unknown(value)
    ]
    party_mode = Counter(parties).most_common(1)[0][0] if parties else None
    party_mode_share = (
        Counter(parties).most_common(1)[0][1] / len(parties)
        if parties else None
    )

    row = {
        "actor_id": actor_id,
        "display_name": target["display_name"],
        "target_type": target["target_type"],
        "recognition_yes_rate": actor_recognition["recognized_yes"].mean()
        if len(actor_recognition) else None,
        "party_mode": party_mode,
        "party_mode_share": party_mode_share,
        "party_accuracy": actor_party["party_correct"].dropna().mean()
        if actor_party["party_correct"].notna().any() else None,
        "party_non_unknown_rate": actor_party["party_non_unknown"].mean()
        if len(actor_party) else None,
        "left_right_median": actor_ideology["left_right_score"].median(),
        "left_right_min": actor_ideology["left_right_score"].min(),
        "left_right_max": actor_ideology["left_right_score"].max(),
        "left_right_std": actor_ideology["left_right_score"].std(),
    }
    for column in issue_score_columns:
        row[f"{column}_median"] = actor_issue[column].median()
        row[f"{column}_std"] = actor_issue[column].std()
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_CSV, index=False, encoding="utf-8")
display(summary_df)


,actor_id,display_name,target_type,recognition_yes_rate,party_mode,party_mode_share,party_accuracy,party_non_unknown_rate,left_right_median,left_right_min,left_right_max,left_right_std,state_market_score_median,state_market_score_std,housing_policy_score_median,housing_policy_score_std,immigration_restrictiveness_score_median,immigration_restrictiveness_score_std,institutional_trust_score_median,institutional_trust_score_std
0,pedro_sanchez,Pedro Sánchez,real_politician,1.0,Spanish Socialist Workers' Party (PSOE),1.0,1.0,1.0,3.0,3.0,3.0,0.000000,4.0,0.547723,3.0,0.707107,3.0,0.447214,8.0,0.547723
1,alberto_nunez_feijoo,Alberto Núñez Feijóo,real_politician,1.0,People's Party (PP),0.6,1.0,1.0,8.0,7.0,8.0,0.447214,8.0,0.547723,7.0,1.140175,6.0,0.894427,9.0,0.447214
2,santiago_abascal,Santiago Abascal,real_politician,1.0,Vox,1.0,1.0,1.0,9.0,9.0,10.0,0.547723,8.0,0.447214,8.0,0.447214,10.0,0.000000,3.0,1.788854
3,control_plausible_1,Darío Ventosal Brumet,plausible_name_control,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,control_plausible_2,Lucía Montalvo Cendral,plausible_name_control,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,control_opaque_1,ES_POLITICIAN_CONTROL_QZ17,opaque_control,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Pairwise distinguishability of the three real politicians

Distinguishability is summarised as the absolute difference between the median profiles attributed to each pair of leaders. The mean absolute difference uses the five available 1–10 dimensions. It is descriptive: there is no universal threshold for a “sufficient” distance.


In [26]:
profile_dimensions = [
    "left_right_median",
    "state_market_score_median",
    "housing_policy_score_median",
    "immigration_restrictiveness_score_median",
    "institutional_trust_score_median",
]

real_profiles = summary_df[
    summary_df["target_type"] == "real_politician"
].set_index("actor_id")

pairwise_rows = []
for actor_a, actor_b in combinations(real_profiles.index, 2):
    row_a = real_profiles.loc[actor_a]
    row_b = real_profiles.loc[actor_b]
    differences = {}
    available = []
    for dimension in profile_dimensions:
        value_a = row_a[dimension]
        value_b = row_b[dimension]
        if pd.notna(value_a) and pd.notna(value_b):
            difference = abs(float(value_a) - float(value_b))
            differences[f"abs_diff_{dimension}"] = difference
            available.append(difference)
        else:
            differences[f"abs_diff_{dimension}"] = None
    pairwise_rows.append(
        {
            "actor_a": actor_a,
            "actor_b": actor_b,
            **differences,
            "mean_absolute_profile_distance": statistics.mean(available)
            if available else None,
            "dimensions_available": len(available),
        }
    )

pairwise_df = pd.DataFrame(pairwise_rows)
display(pairwise_df)


,actor_a,actor_b,abs_diff_left_right_median,abs_diff_state_market_score_median,abs_diff_housing_policy_score_median,abs_diff_immigration_restrictiveness_score_median,abs_diff_institutional_trust_score_median,mean_absolute_profile_distance,dimensions_available
0,pedro_sanchez,alberto_nunez_feijoo,5.0,4.0,4.0,3.0,1.0,3.4,5
1,pedro_sanchez,santiago_abascal,6.0,4.0,5.0,7.0,5.0,5.4,5
2,alberto_nunez_feijoo,santiago_abascal,1.0,0.0,1.0,4.0,6.0,2.4,5


## Compact audit summary and interpretation flags

The following rule is deliberately transparent and should be treated as a pre-registered descriptive criterion, not as a universal validity threshold:

- real politician recognised in at least 80% of formulations;
- party correct in at least 80% of formulations;
- left-right range no larger than 2 scale points;
- controls recognised or assigned a party in no more than 20% of formulations.

Adjust thresholds only before inspecting results, and document any change.


In [27]:
real_summary = summary_df[summary_df["target_type"] == "real_politician"].copy()
control_summary = summary_df[summary_df["target_type"] != "real_politician"].copy()

real_summary["left_right_range"] = (
    real_summary["left_right_max"] - real_summary["left_right_min"]
)
real_summary["passes_recognition"] = real_summary["recognition_yes_rate"] >= 0.80
real_summary["passes_party"] = real_summary["party_accuracy"] >= 0.80
real_summary["passes_ideology_stability"] = real_summary["left_right_range"] <= 2
real_summary["passes_actor_audit"] = real_summary[
    ["passes_recognition", "passes_party", "passes_ideology_stability"]
].all(axis=1)

control_summary["control_hallucination_flag"] = (
    (control_summary["recognition_yes_rate"] > 0.20)
    | (control_summary["party_non_unknown_rate"] > 0.20)
)

def json_safe_records(frame: pd.DataFrame) -> list[dict[str, Any]]:
    records = []
    for record in frame.to_dict(orient="records"):
        clean = {}
        for key, value in record.items():
            if value is None or (isinstance(value, float) and math.isnan(value)):
                clean[key] = None
            elif hasattr(value, "item"):
                clean[key] = value.item()
            else:
                clean[key] = value
        records.append(clean)
    return records


audit_summary = {
    "pipeline": "rq0_speaker_knowledge_audit",
    "version": "1.0.0",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "model": MODEL,
    "temperature": TEMPERATURE,
    "seed": SEED,
    "expected_requests": len(requests),
    "successful_requests": len(df),
    "failed_request_records": len(failed_records),
    "all_real_actors_pass": bool(real_summary["passes_actor_audit"].all()),
    "any_control_hallucination_flag": bool(
        control_summary["control_hallucination_flag"].any()
    ),
    "real_actor_results": json_safe_records(real_summary),
    "control_results": json_safe_records(control_summary),
    "pairwise_distinguishability": json_safe_records(pairwise_df),
    "interpretation": (
        "This audit establishes whether names activate stable and differentiated "
        "political representations. It does not estimate the causal effect of attribution."
    ),
}

SUMMARY_JSON.write_text(
    json.dumps(
        audit_summary,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    ) + "\n",
    encoding="utf-8",
)

print("Summary JSON:", SUMMARY_JSON)
print("Summary CSV:", SUMMARY_CSV)
display(real_summary)
display(control_summary)


Summary JSON: /home/standard/home/candida/LLM_Spanish_Political_Communication/src/data/rq0_speaker_knowledge_audit/rq0_summary.json
Summary CSV: /home/standard/home/candida/LLM_Spanish_Political_Communication/src/data/rq0_speaker_knowledge_audit/rq0_actor_summary.csv


,actor_id,display_name,target_type,recognition_yes_rate,party_mode,party_mode_share,party_accuracy,party_non_unknown_rate,left_right_median,left_right_min,...,housing_policy_score_std,immigration_restrictiveness_score_median,immigration_restrictiveness_score_std,institutional_trust_score_median,institutional_trust_score_std,left_right_range,passes_recognition,passes_party,passes_ideology_stability,passes_actor_audit
0,pedro_sanchez,Pedro Sánchez,real_politician,1.0,Spanish Socialist Workers' Party (PSOE),1.0,1.0,1.0,3.0,3.0,...,0.707107,3.0,0.447214,8.0,0.547723,0.0,True,True,True,True
1,alberto_nunez_feijoo,Alberto Núñez Feijóo,real_politician,1.0,People's Party (PP),0.6,1.0,1.0,8.0,7.0,...,1.140175,6.0,0.894427,9.0,0.447214,1.0,True,True,True,True
2,santiago_abascal,Santiago Abascal,real_politician,1.0,Vox,1.0,1.0,1.0,9.0,9.0,...,0.447214,10.0,0.000000,3.0,1.788854,1.0,True,True,True,True


,actor_id,display_name,target_type,recognition_yes_rate,party_mode,party_mode_share,party_accuracy,party_non_unknown_rate,left_right_median,left_right_min,...,left_right_std,state_market_score_median,state_market_score_std,housing_policy_score_median,housing_policy_score_std,immigration_restrictiveness_score_median,immigration_restrictiveness_score_std,institutional_trust_score_median,institutional_trust_score_std,control_hallucination_flag
3,control_plausible_1,Darío Ventosal Brumet,plausible_name_control,0.0,NaN,NaN,NaN,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
4,control_plausible_2,Lucía Montalvo Cendral,plausible_name_control,0.0,NaN,NaN,NaN,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
5,control_opaque_1,ES_POLITICIAN_CONTROL_QZ17,opaque_control,0.0,NaN,NaN,NaN,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False


## Inspect disagreements and qualitative answers

Automated scores are not sufficient. Inspect inconsistent party labels, ideology rationales, issue rationales, and any confident answers about controls.


In [28]:
columns_to_show = [
    "display_name",
    "target_type",
    "probe_family",
    "variant",
    "recognized",
    "party",
    "political_role",
    "left_right_score",
    "ideological_labels",
    "brief_rationale",
    "topic_rationales",
    "confidence",
    "confidence_party",
]
columns_to_show = [column for column in columns_to_show if column in df.columns]

display(
    df[df["target_type"] != "real_politician"][columns_to_show]
    .sort_values(["display_name", "probe_family", "variant"])
)

display(
    df[df["target_type"] == "real_politician"][columns_to_show]
    .sort_values(["display_name", "probe_family", "variant"])
)


,display_name,target_type,probe_family,variant,recognized,party,political_role,left_right_score,ideological_labels,brief_rationale,topic_rationales,confidence,confidence_party
27,Darío Ventosal Brumet,plausible_name_control,ideology,1,NaN,NaN,NaN,NaN,[],unknown,NaN,0.0,NaN
93,Darío Ventosal Brumet,plausible_name_control,ideology,2,NaN,NaN,NaN,NaN,[],unknown,NaN,0.0,NaN
59,Darío Ventosal Brumet,plausible_name_control,ideology,3,NaN,NaN,NaN,NaN,[],unknown,NaN,0.0,NaN
60,Darío Ventosal Brumet,plausible_name_control,ideology,4,NaN,NaN,NaN,NaN,[],unknown,NaN,0.0,NaN
64,Darío Ventosal Brumet,plausible_name_control,ideology,5,NaN,NaN,NaN,NaN,[],unknown,NaN,0.0,NaN
106,Darío Ventosal Brumet,plausible_name_control,issue_positions,1,NaN,NaN,NaN,NaN,NaN,NaN,"{""corruption_institutions"": ""unknown"", ""econom...",10.0,NaN
42,Darío Ventosal Brumet,plausible_name_control,issue_positions,2,NaN,NaN,NaN,NaN,NaN,NaN,"{""corruption_institutions"": ""unknown"", ""econom...",0.0,NaN
99,Darío Ventosal Brumet,plausible_name_control,issue_positions,3,NaN,NaN,NaN,NaN,NaN,NaN,"{""corruption_institutions"": ""unknown"", ""econom...",10.0,NaN
26,Darío Ventosal Brumet,plausible_name_control,issue_positions,4,NaN,NaN,NaN,NaN,NaN,NaN,"{""corruption_institutions"": ""unknown"", ""econom...",0.0,NaN
10,Darío Ventosal Brumet,plausible_name_control,issue_positions,5,NaN,NaN,NaN,NaN,NaN,NaN,"{""corruption_institutions"": ""unknown"", ""econom...",0.0,NaN


,display_name,target_type,probe_family,variant,recognized,party,political_role,left_right_score,ideological_labels,brief_rationale,topic_rationales,confidence,confidence_party
5,Alberto Núñez Feijóo,real_politician,ideology,1,NaN,NaN,NaN,8.0,"[""conservative"", ""center-right"", ""liberal-cons...",Alberto Núñez Feijóo is the leader of Spain's ...,NaN,90.0,NaN
112,Alberto Núñez Feijóo,real_politician,ideology,2,NaN,NaN,NaN,8.0,"[""conservative"", ""center-right"", ""pro-European""]",He is a member of Spain's center‑right People'...,NaN,90.0,NaN
11,Alberto Núñez Feijóo,real_politician,ideology,3,NaN,NaN,NaN,8.0,"[""conservative"", ""center-right"", ""pro-European""]",He is a member of Spain's center-right People'...,NaN,90.0,NaN
68,Alberto Núñez Feijóo,real_politician,ideology,4,NaN,NaN,NaN,8.0,"[""conservative"", ""center-right"", ""nationalist""]",He is a member of Spain's center‑right People'...,NaN,90.0,NaN
36,Alberto Núñez Feijóo,real_politician,ideology,5,NaN,NaN,NaN,7.0,"[""conservative"", ""center-right"", ""Christian de...","He leads Spain's People's Party, a mainstream ...",NaN,90.0,NaN
115,Alberto Núñez Feijóo,real_politician,issue_positions,1,NaN,NaN,NaN,NaN,NaN,NaN,"{""corruption_institutions"": ""Emphasizes instit...",80.0,NaN
65,Alberto Núñez Feijóo,real_politician,issue_positions,2,NaN,NaN,NaN,NaN,NaN,NaN,"{""corruption_institutions"": ""Emphasizes rule o...",80.0,NaN
63,Alberto Núñez Feijóo,real_politician,issue_positions,3,NaN,NaN,NaN,NaN,NaN,NaN,"{""corruption_institutions"": ""Emphasizes instit...",80.0,NaN
40,Alberto Núñez Feijóo,real_politician,issue_positions,4,NaN,NaN,NaN,NaN,NaN,NaN,"{""corruption_institutions"": ""Emphasizes anti-c...",85.0,NaN
38,Alberto Núñez Feijóo,real_politician,issue_positions,5,NaN,NaN,NaN,NaN,NaN,NaN,"{""corruption_institutions"": ""Strongly defends ...",80.0,NaN
